## Assignment 3: $k$ Nearest Neighbor


**Q1.**
1. What is the difference between regression and classification?
2. What is a confusion table? What does it help us understand about a model's performance?
3. What does the SSE quantify about a particular model?
4. What are overfitting and underfitting? 
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Adeline Kwakye

**Date:** 9/28/2026

1. Classifiction predicts categories or labels. You use it when you want to categorize data into different groups. Regression predicts continuous values like price, or temperature. It identifies trends and patterns in data and determines the relationship between independent variables and a continuous target variable.

2. A confusion table will count predicitions by actual class and predicted class, where the diagonal cells count correct predictions and off-diagonal cells count misclassifictions. It helps us understand the accuracy of a model as well as what it gets confused about.

3. The SSE quantifies how much error a model makes in the validation set.

4. Underfitting means the model is too simple and doesn't cover all real patterns in the data. Overfitting means that the model learns not just underlying pattern, but also noise or random quirks in the training data.

5. Splitting the data helps us see whether the model is actually learning a general pattern or just memorizing the data it was trained on. Using SSE or validation accuracy to choose K helps pick a model that will perform well on new observations. 

6. Reporting a class label as a prediction is good because it's easy to use and understand but it doesn't tell you how confident the model is in that decision. Reporting a class label as a probability distribution is good beause it shows all possible classes and shows how certain and uncertain the model is. However, the probabilities might not be perfect and applicable to every situation.

In [1]:
# Your Phase 1 solution to the problem goes here.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

df = pd.read_csv('data/land_mines.csv')
# performing eda on target label and features
print( "Summary of target label\n")
#summary on target label
summary = pd.DataFrame({
    "mean": [df["mine_type"].mean()],
    "median" :[df["mine_type"].median()],
    "mode" : [df["mine_type"].mode().iloc[0]],
    "Q1": [df["mine_type"].quantile(0.25)],
    "Q3": [df["mine_type"].quantile(0.75)],
})

display(summary.round(3))
#summary of voltage
print("Summary of voltage\n")
summary_v = pd.DataFrame({
    "mean": [df["voltage"].mean()],
    "median" :[df["voltage"].median()],
    "mode" : [df["voltage"].mode().iloc[0]],
    "Q1": [df["voltage"].quantile(0.25)],
    "Q3": [df["voltage"].quantile(0.75)],
})
display(summary_v.round(3))

#summary of height
print("Summary of height\n")
summary_h = pd.DataFrame({
    "mean": [df["height"].mean()],
    "median" :[df["height"].median()],
    "mode" : [df["height"].mode().iloc[0]],
    "Q1": [df["height"].quantile(0.25)],
    "Q3": [df["height"].quantile(0.75)],
})
display(summary_h.round(3))

# summary of soil
print("Summary of soil\n")
summary_s = pd.DataFrame({
    "mean": [df["soil"].mean()],
    "median" :[df["soil"].median()],
    "mode" : [df["soil"].mode().iloc[0]],
    "Q1": [df["soil"].quantile(0.25)],
    "Q3": [df["soil"].quantile(0.75)],
})
display(summary_s.round(3))

# split the data
print("split data 50/50\n")
X = df[["voltage", "height", "soil"]] # features
y = df["mine_type"]

(X_train, X_valid, y_train, y_valid,) = train_test_split(X, y, test_size = 0.50, random_state = 65 )
# check the size
print("Training:\n")
print(y_train.value_counts())
print("Validation:\n")
print(y_valid.value_counts())

# scaler
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# build KNN
print("Building k-NN Classifier\n")
evals_ks = np.arange(1, 51)
eval_valid_accuracy = []
# iterate through each K
for eval_k in evals_ks:
    eval_candidate = KNeighborsClassifier(n_neighbors=int(eval_k))
    eval_candidate.fit(X_train_scaled, y_train)
    eval_valid_hat = eval_candidate.predict(X_valid_scaled)
    eval_valid_accuracy.append(
        accuracy_score(y_valid,eval_valid_hat)
    )
eval_k_star = int(evals_ks[np.argmax(eval_valid_accuracy)]) 
print("highest k value: ", eval_k_star)
classification_model = KNeighborsClassifier(n_neighbors=eval_k_star).fit(
    X_train_scaled, y_train
)
y_valid_predict = classification_model.predict(X_valid_scaled) # predicted mine_type for each observation

# confusion maxtrix
print("confusion matrix\n")
cm = confusion_matrix(y_valid, y_valid_predict)
print(cm)
 # check accuracy
accuracy = accuracy_score(y_valid, y_valid_predict)
print("Accuracy:", accuracy)

Summary of target label



,mean,median,mode,Q1,Q3
0,2.953,3.0,1,2.0,4.0


Summary of voltage



,mean,median,mode,Q1,Q3
0,0.431,0.36,1.0,0.31,0.483


Summary of height



,mean,median,mode,Q1,Q3
0,0.509,0.545,0.182,0.273,0.727


Summary of soil



,mean,median,mode,Q1,Q3
0,0.504,0.6,0.0,0.2,0.8


split data 50/50

Training:

mine_type
1    37
5    36
3    36
2    32
4    28
Name: count, dtype: int64
Validation:

mine_type
4    38
2    38
1    34
3    30
5    29
Name: count, dtype: int64
Building k-NN Classifier

highest k value:  2
confusion matrix

[[21  1  6  4  2]
 [ 0 33  2  3  0]
 [ 5  5 10  6  4]
 [13  5 10  9  1]
 [10  1  8  7  3]]
Accuracy: 0.44970414201183434


3. I selected K by testing a range of K values and comparing the validation accuracy. I wanted the K value that had the highest validation accuracy. I did a range of K values from 1 through 50 and K = 2 produced the highest validation accuracy.
4. The model had an overall accuracy of about 44.97%. Mine type 2 was found to be the most accurate with 33 of 38 observations correctly classified, equating to 86.8%. Mine types 4 and 5 were less accurate. For type 4, only 9 of 38 observations were correctly classified, equating to 23.7%. For type 5 only 3 of 29 observations were correctly classified, equating to 10.3%.

5. I would advise someone to not use this as the only method to determine which land mines to use. While it's able to correctly classify some mines, the accuracy overall was only 44.97%. Meaning that it's not even accurate about 50% of the time. With the confusion matrix we saw that types 4 and 5 were extremely misclassified. This model should be used in addition to other identification methods as misclassifiying mines could have dangerous consequences.

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** 4b26a78ad7ad8c7dae541fdf1ae881dd0c807324
- **AI tool and model used:** Claude Sonnet 5.5

### *Prompts used

1. hi claude can you look over my homework questions and code and give me any corrections and tips that i need to make. I'll attach the homework write up and my answers


### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. EDA on mine_type isn't meaningful. Mean, median, and quartiles of a categorical label (1 to 5) don't tell you anything. Use class counts and proportions instead:

print(df["mine_type"].value_counts().sort_index())
print(df["mine_type"].value_counts(normalize=True).sort_index().round(3))

For the features, df.describe() replaces your three repeated blocks. Add histograms or boxplots of each feature by mine_type, and check for missing values. Also check df["soil"].unique(). If it only takes a few discrete values, it's really categorical, and its mean isn't meaningful either. Since you imported seaborn, a pairplot colored by mine_type would be a quick win.

2. Stratify your split. Add stratify=y to train_test_split so every mine type appears in both halves in the same proportion.

3. Label your confusion matrix and add per-class metrics. A bare array is hard to read:

python
from sklearn.metrics import classification_report
labels = sorted(y.unique())
print(pd.DataFrame(cm, index=[f"actual {l}" for l in labels],
                   columns=[f"pred {l}" for l in labels]))
print(classification_report(y_valid, y_valid_predict))

4. Q2.5 is where you lose the most. The prompt says you can have many accurate predictions for a type and still make many mistakes. That's the recall vs. precision distinction. Recall is the row-wise view, and precision is the column-wise view: when the model says "type 2," how often is it right? Look at the type 2 column of your matrix. If lots of other types get predicted as 2, then high recall for type 2 can coexist with many wrong "type 2" calls. Your answer should also:

Correct "not even accurate about 50% of the time." With 5 classes, 45% is more than double chance, so the model has real signal. It just isn't reliable enough to act on alone.
Include a concrete recommendation: use predict_proba rather than just labels, flag low-confidence predictions for human or manual verification, and note that errors have unequal costs (treating a dangerous mine as a harmless type is worse than the reverse).
Fix the phrase "which land mines to use," which I assume means "identify" or "handle."


### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Modify | I decided to modify the suggestions provided by Claude. Claude made a good point in saying how it's more appropriate to do class counts and proportions for the categorical target mine type. I don't think it's necessary to accept every type of summary method it proposed but i think using methods that are better suited for a categorical target is more accurated as opposed to doing methods that work better on quantitative.|
| 2 | Accept| I will accept this suggestion because after looking into stratify=y, it serves the purpose of ensuring that the training and validation set have the same proportion of each mine type. |
| 3 | Accept | I will accept this suggestion because I don't think it can hurt having labels on the matrix which can help with the readability of it. |
| 4 | Reject | While I do think that what Claude suggests makes sense, I'm going to reject this suggestion. I think it goes beyond the scope of what question 5 was asking to begin with. Question 5 was just saying given the mistakes that it makes how would advise someone to use this model. I think it overcomplicated the answer more than necessary. |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

1. Classifiction predicts categories or labels. You use it when you want to categorize data into different groups. Regression predicts continuous values like price, or temperature. It identifies trends and patterns in data and determines the relationship between independent variables and a continuous target variable.

2. A confusion table will count predicitions by actual class and predicted class, where the diagonal cells count correct predictions and off-diagonal cells count misclassifictions. It helps us understand the accuracy of a model as well as what it gets confused about.

3. The SSE quantifies how much error a model makes in the validation set.

4. Underfitting means the model is too simple and doesn't cover all real patterns in the data. Overfitting means that the model learns not just underlying pattern, but also noise or random quirks in the training data.

5. Splitting the data helps us see whether the model is actually learning a general pattern or just memorizing the data it was trained on. Using SSE or validation accuracy to choose K helps pick a model that will perform well on new observations. 

6. Reporting a class label as a prediction is good because it's easy to use and understand but it doesn't tell you how confident the model is in that decision. Reporting a class label as a probability distribution is good beause it shows all possible classes and shows how certain and uncertain the model is. However, the probabilities might not be perfect and applicable to every situation.

In [6]:
# Your Phase 1 solution to the problem goes here.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

df = pd.read_csv('data/land_mines.csv')
# performing eda on target label and features
# print( "Summary of target label\n")

#summary on target label


#summary = pd.DataFrame({
#   "mean": [df["mine_type"].mean()],
#    "median" :[df["mine_type"].median()],
#    "mode" : [df["mine_type"].mode().iloc[0]],
#    "Q1": [df["mine_type"].quantile(0.25)],
#    "Q3": [df["mine_type"].quantile(0.75)],
#})

print("Class counts for mine type\n")
print(df["mine_type"].value_counts().sort_index())
print("\nClass proportions for mine type\n")
print(df["mine_type"].value_counts(normalize=True).sort_index().round(3))

#display(summary.round(3))
#summary of voltage
print("Summary of voltage\n")
summary_v = pd.DataFrame({
    "mean": [df["voltage"].mean()],
    "median" :[df["voltage"].median()],
    "mode" : [df["voltage"].mode().iloc[0]],
    "Q1": [df["voltage"].quantile(0.25)],
    "Q3": [df["voltage"].quantile(0.75)],
})
display(summary_v.round(3))

#summary of height
print("Summary of height\n")
summary_h = pd.DataFrame({
    "mean": [df["height"].mean()],
    "median" :[df["height"].median()],
    "mode" : [df["height"].mode().iloc[0]],
    "Q1": [df["height"].quantile(0.25)],
    "Q3": [df["height"].quantile(0.75)],
})
display(summary_h.round(3))

# summary of soil
print("Summary of soil\n")
summary_s = pd.DataFrame({
    "mean": [df["soil"].mean()],
    "median" :[df["soil"].median()],
    "mode" : [df["soil"].mode().iloc[0]],
    "Q1": [df["soil"].quantile(0.25)],
    "Q3": [df["soil"].quantile(0.75)],
})
display(summary_s.round(3))

# split the data
print("split data 50/50\n")
X = df[["voltage", "height", "soil"]] # features
y = df["mine_type"]

(X_train, X_valid, y_train, y_valid,) = train_test_split(X, y, test_size = 0.50, random_state = 65, stratify=y )
# check the size
print("Training:\n")
print(y_train.value_counts())
print("Validation:\n")
print(y_valid.value_counts())

# scaler
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# build KNN
print("Building k-NN Classifier\n")
evals_ks = np.arange(1, 51)
eval_valid_accuracy = []
# iterate through each K
for eval_k in evals_ks:
    eval_candidate = KNeighborsClassifier(n_neighbors=int(eval_k))
    eval_candidate.fit(X_train_scaled, y_train)
    eval_valid_hat = eval_candidate.predict(X_valid_scaled)
    eval_valid_accuracy.append(
        accuracy_score(y_valid,eval_valid_hat)
    )
eval_k_star = int(evals_ks[np.argmax(eval_valid_accuracy)]) 
print("highest k value: ", eval_k_star)
classification_model = KNeighborsClassifier(n_neighbors=eval_k_star).fit(
    X_train_scaled, y_train
)
y_valid_predict = classification_model.predict(X_valid_scaled) # predicted mine_type for each observation

# confusion maxtrix
print("confusion matrix\n")
cm = confusion_matrix(y_valid, y_valid_predict)
labels = sorted(y.unique())
print(pd.DataFrame(
    cm,
    index=[f"actual {l}" for l in labels],
    columns=[f"pred {l}" for l in labels]
))

 # check accuracy
accuracy = accuracy_score(y_valid, y_valid_predict)
print("Accuracy:", accuracy)

Class counts for mine type

mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64

Class proportions for mine type

mine_type
1    0.210
2    0.207
3    0.195
4    0.195
5    0.192
Name: proportion, dtype: float64
Summary of voltage



,mean,median,mode,Q1,Q3
0,0.431,0.36,1.0,0.31,0.483


Summary of height



,mean,median,mode,Q1,Q3
0,0.509,0.545,0.182,0.273,0.727


Summary of soil



,mean,median,mode,Q1,Q3
0,0.504,0.6,0.0,0.2,0.8


split data 50/50

Training:

mine_type
2    35
1    35
4    33
5    33
3    33
Name: count, dtype: int64
Validation:

mine_type
1    36
2    35
3    33
4    33
5    32
Name: count, dtype: int64
Building k-NN Classifier

highest k value:  2
confusion matrix

          pred 1  pred 2  pred 3  pred 4  pred 5
actual 1      27       0       5       2       2
actual 2       0      31       2       1       1
actual 3      10       2      10      10       1
actual 4      15       6       3       9       0
actual 5      15       0      10       5       2
Accuracy: 0.46745562130177515


3. I selected K by testing a range of K values and comparing the validation accuracy. I wanted the K value that had the highest validation accuracy. I did a range of K values from 1 through 50 and K = 2 produced the highest validation accuracy.
4. The model had an overall accuracy of about 44.97%. Mine type 2 was found to be the most accurate with 33 of 38 observations correctly classified, equating to 86.8%. Mine types 4 and 5 were less accurate. For type 4, only 9 of 38 observations were correctly classified, equating to 23.7%. For type 5 only 3 of 29 observations were correctly classified, equating to 10.3%.

5. I would advise someone to not use this as the only method to determine which land mines to use. While it's able to correctly classify some mines, the accuracy overall was only 44.97%. Meaning that it's not even accurate about 50% of the time. With the confusion matrix we saw that types 4 and 5 were extremely misclassified. This model should be used in addition to other identification methods as misclassifiying mines could have dangerous consequences.

### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]

The main improvements that AI suggested for me was helping me determine when to use categorical summarization as opposed to using quantitative summarization, readability improvements, as well as extra measures to ensure the accuracy of my code. Some limitations with AI is sometimes it has the tendency to overcomplicate certain answers or read more into what the question asking. Unless I had the wrong interpretation of what question 2.5 was asking, I felt as if the answer it provided, while good, I don't think it was entirely applicable for this specific question. I think it overcomplicated what the question was truly asking so I wasn't able to really use much of that suggestion. I verified the answers that it gave me by doing a google search and looking on stack overflow to see if the suggested method was accurate and a method that actually worked. My only concern would be with Question 2.5, and wondering if maybe I was simplifying what the question was asking and in reality what Claude was suggesting was the actual approach that the professor was looking for. 